# Many-body perturbation theory

Companion notebook to Chapter 9 of *Quantum mechanics for many-particle
systems*.  Every number quoted in the chapter is produced here; the code is
the same as in `BookManybody/BookMaterial/Programs/mbpt.py`.

Everything is done in the determinant basis of Chapter 5, where $\hat H_0$ is
diagonal and the perturbation is the rest of the matrix.  With
$\hat R = \hat Q/(W_0-\hat H_0)$ the Rayleigh-Schrödinger series follows from

$$|\Psi^{(n)}\rangle = \hat R\Big[\hat H_I|\Psi^{(n-1)}\rangle
   - \sum_{k=1}^{n-1}\Delta E^{(k)}|\Psi^{(n-k)}\rangle\Big],
\qquad
\Delta E^{(n+1)} = \langle\Phi_0|\hat H_I|\Psi^{(n)}\rangle,$$

which reproduces the explicit first, second and third-order expressions and
continues to any order.

Contents:

1. The recursion against the explicit expressions
2. The pairing model
3. Higher order: does the series converge?
4. The Lipkin model
5. Size extensivity: Rayleigh-Schrödinger against Brillouin-Wigner
6. Brillouin-Wigner and its exact resummation
7. The Hubbard model
8. The pairing plus particle-hole model
9. Does the partition matter?
10. Everything against everything

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join("..", "BookManybody", "BookMaterial", "Programs"))
import mbpt

np.set_printoptions(precision=6, suppress=True, linewidth=120)

## 1. The recursion against the explicit expressions

$$\Delta E^{(1)} = V_{00},\qquad
\Delta E^{(2)} = \sum_{m\neq0}\frac{V_{0m}V_{m0}}{D_m},$$
$$\Delta E^{(3)} = \sum_{m,n\neq0}\frac{V_{0m}V_{mn}V_{n0}}{D_mD_n}
  - V_{00}\sum_{m\neq0}\frac{V_{0m}V_{m0}}{D_m^2}.$$

The second term of the third-order energy — the *renormalisation* term — is
the one to watch.  It is what makes the series size extensive.

In [ ]:
mbpt.demo_recursion()

## 2. The pairing model

Four doubly degenerate levels, four particles, $\xi=1$.  The energy through
first order is $2-g$, which is exactly the Hartree-Fock energy of Chapter 6 —
for this interaction the mean field does nothing, so everything beyond first
order is genuine correlation energy.

In [ ]:
mbpt.demo_pairing()

In [ ]:
import rpa as rpamod
import fci

gs = np.linspace(0.1, 2.2, 22)
fock = rpamod.FockSpace(4)
ref, m2, m3, cid, rpa_e, exact = [], [], [], [], [], []
for g in gs:
    part, model = mbpt.pairing_partition(g=g)
    e = mbpt.rayleigh_schrodinger(part, order=3)
    r0 = part.reference_energy
    ref.append(r0); m2.append(r0 + e[1]); m3.append(r0 + e[1] + e[2])
    cid.append(model.energies(2)[0])
    rr = rpamod.tda_rpa(fock, 4, g, 0.0)
    rpa_e.append(rr["hf"] + rr["ecorr"])
    exact.append(part.exact())

fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))
ax[0].plot(gs, exact, "k-o", ms=3, label="exact (FCI)")
ax[0].plot(gs, ref, "C7--", label="Hartree-Fock")
ax[0].plot(gs, m2, "C0-", label="MBPT2")
ax[0].plot(gs, m3, "C1-", label="MBPT3")
ax[0].plot(gs, cid, "C2-", label="CID")
ax[0].plot(gs, rpa_e, "C3-", label="HF + RPA")
ax[0].set_xlabel("$g$"); ax[0].set_ylabel("ground-state energy")
ax[0].legend(fontsize=8)

for y, lab, c in ((m2, "MBPT2", "C0"), (m3, "MBPT3", "C1"),
                  (cid, "CID", "C2"), (rpa_e, "HF + RPA", "C3")):
    ax[1].semilogy(gs, np.abs(np.array(y) - np.array(exact)), c, label=lab)
ax[1].set_xlabel("$g$"); ax[1].set_ylabel("|error|")
ax[1].set_title("no method wins everywhere")
ax[1].legend(fontsize=8)
fig.tight_layout(); plt.show()

The error panel is the interesting one.  At small $g$ third-order perturbation
theory is best by orders of magnitude and RPA is the worst; by $g=2$ the
ordering has reversed completely.

## 3. Higher order: does the series converge?

The recursion continues to any order, so we can look rather than guess.

In [ ]:
mbpt.demo_orders()

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.6))
for g, style in ((0.5, "-o"), (1.0, "-s"), (1.5, "-^"), (2.0, "-v")):
    part, _ = mbpt.pairing_partition(g=g)
    e = np.abs(np.array(mbpt.rayleigh_schrodinger(part, order=41)))
    n = np.arange(1, len(e) + 1)
    ax.semilogy(n[1:], np.maximum(e[1:], 1e-30), style, ms=3,
                label=f"$g = {g}$")
ax.set_xlabel("order $n$"); ax.set_ylabel(r"$|\Delta E^{(n)}|$")
ax.set_title("convergent below, divergent above")
ax.legend(); fig.tight_layout(); plt.show()

At $g=0.5$ and $g=1$ the terms fall away steadily.  At $g=2$ they grow without
limit.  The radius of convergence lies between — and crucially, **the first
three orders give no warning**: at $g=1.5$ they look perfectly reasonable.

## 4. The Lipkin model

Here $\Delta E^{(1)} = 0$ and $\Delta E^{(3)} = 0$, both identically, because
the interaction changes $J_z$ by exactly two units.  Second order is available
in closed form, $\Delta E^{(2)} = -N(N-1)V^2/4\varepsilon$.

In [ ]:
mbpt.demo_lipkin()

## 5. Size extensivity

Two identical, non-interacting copies of a three-level pairing system.  Any
acceptable method must give exactly twice the energy of one subsystem.
Rayleigh-Schrödinger is exact at every order; Brillouin-Wigner is not.

In [ ]:
mbpt.demo_size_extensivity()

In [ ]:
gs = np.linspace(0.1, 1.2, 12)
err_rs, err_bw = [], []
for g in gs:
    one = mbpt.IndependentPairing(subsystems=1, levels=3, g=g).partition()
    two = mbpt.IndependentPairing(subsystems=2, levels=3, g=g).partition()
    e1 = mbpt.rayleigh_schrodinger(one, order=3)
    e2 = mbpt.rayleigh_schrodinger(two, order=3)
    err_rs.append(abs(sum(e2) - 2 * sum(e1)))
    b1, _ = mbpt.brillouin_wigner(one, order=2)
    b2, _ = mbpt.brillouin_wigner(two, order=2)
    err_bw.append(abs(b2 - 2 * b1))

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.semilogy(gs, np.maximum(err_rs, 1e-18), "o-", label="Rayleigh-Schrödinger")
ax.semilogy(gs, err_bw, "s-", label="Brillouin-Wigner (2nd order)")
ax.set_xlabel("$g$"); ax.set_ylabel("|E(AB) - 2 E(A)|")
ax.set_title("size-extensivity error")
ax.legend(); fig.tight_layout(); plt.show()

## 6. Brillouin-Wigner and its exact resummation

$$\Delta E = \langle\Phi_0|\hat H_I
  + \hat H_I\hat Q\,[E-\hat H_0-\hat Q\hat H_I\hat Q]^{-1}\hat Q\hat H_I
  |\Phi_0\rangle$$

is not an approximation at all — solved self-consistently it must reproduce
the exact energy, and it does.

In [ ]:
mbpt.demo_brillouin_wigner()

## 7. The Hubbard model

Six sites at half filling in the momentum basis.  The third-order energy
vanishes identically here — special to this filling and this interaction, not
a general feature — so fourth order is shown as well.

In [ ]:
mbpt.demo_hubbard()

## 8. The pairing plus particle-hole model

The particle-hole term breaks pairs, so the reference couples to singles as
well as doubles.  We split the second-order sum by excitation rank.  The
singles piece would vanish identically in a Hartree-Fock basis.

In [ ]:
mbpt.demo_pairing_ph()

## 9. Does the partition matter?

Nothing forced our choice of $\hat H_0$.  For three of the four models the
one-body and Epstein-Nesbet partitions give *identical* answers, because the
diagonal of $\hat H_I$ is constant over every determinant the reference can
reach.  Only the pairing plus particle-hole model distinguishes them.

In [ ]:
mbpt.demo_partitions()

## 10. Everything against everything

In [ ]:
mbpt.demo_comparison()

## The full program

Everything above lives in `BookManybody/BookMaterial/Programs/mbpt.py`, which
runs as a script and prints all ten demonstrations of the chapter.

In [ ]:
print(open(mbpt.__file__).read())